# HyperParameter Tunning

In [2]:
# Imports

import os
import sys

sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath(".."))

from preprocessing import get_X_y
import pandas as pd
from sklearn.model_selection import train_test_split ,RandomizedSearchCV
from sklearn.metrics import r2_score
from scipy.stats import uniform , randint
from xgboost import XGBRegressor

In [3]:
# Data Load

df = pd.read_csv(r"../dataset/KAG_energydata_complete.csv")

df

,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.890000,47.596667,19.200000,44.790000,19.790000,44.730000,19.000000,...,17.033333,45.5300,6.600000,733.5,92.000000,7.000000,63.000000,5.300000,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.890000,46.693333,19.200000,44.722500,19.790000,44.790000,19.000000,...,17.066667,45.5600,6.483333,733.6,92.000000,6.666667,59.166667,5.200000,18.606195,18.606195
2,2016-01-11 17:20:00,50,30,19.890000,46.300000,19.200000,44.626667,19.790000,44.933333,18.926667,...,17.000000,45.5000,6.366667,733.7,92.000000,6.333333,55.333333,5.100000,28.642668,28.642668
3,2016-01-11 17:30:00,50,40,19.890000,46.066667,19.200000,44.590000,19.790000,45.000000,18.890000,...,17.000000,45.4000,6.250000,733.8,92.000000,6.000000,51.500000,5.000000,45.410389,45.410389
4,2016-01-11 17:40:00,60,40,19.890000,46.333333,19.200000,44.530000,19.790000,45.000000,18.890000,...,17.000000,45.4000,6.133333,733.9,92.000000,5.666667,47.666667,4.900000,10.084097,10.084097
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19730,2016-05-27 17:20:00,100,0,25.566667,46.560000,25.890000,42.025714,27.200000,41.163333,24.700000,...,23.200000,46.7900,22.733333,755.2,55.666667,3.333333,23.666667,13.333333,43.096812,43.096812
19731,2016-05-27 17:30:00,90,0,25.500000,46.500000,25.754000,42.080000,27.133333,41.223333,24.700000,...,23.200000,46.7900,22.600000,755.2,56.000000,3.500000,24.500000,13.300000,49.282940,49.282940
19732,2016-05-27 17:40:00,270,10,25.500000,46.596667,25.628571,42.768571,27.050000,41.690000,24.700000,...,23.200000,46.7900,22.466667,755.2,56.333333,3.666667,25.333333,13.266667,29.199117,29.199117
19733,2016-05-27 17:50:00,420,10,25.500000,46.990000,25.414000,43.036000,26.890000,41.290000,24.700000,...,23.200000,46.8175,22.333333,755.2,56.666667,3.833333,26.166667,13.233333,6.322784,6.322784


In [4]:
# Preprocessing

X , y = get_X_y(df)
X = X.drop(["Min" , "Month" , "Day"] , axis=1)

X

,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,T5,...,RH_8,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,Hour
0,30,19.890000,47.596667,19.200000,44.790000,19.790000,44.730000,19.000000,45.566667,17.166667,...,48.900000,17.033333,45.5300,6.600000,733.5,92.000000,7.000000,63.000000,5.300000,17
1,30,19.890000,46.693333,19.200000,44.722500,19.790000,44.790000,19.000000,45.992500,17.166667,...,48.863333,17.066667,45.5600,6.483333,733.6,92.000000,6.666667,59.166667,5.200000,17
2,30,19.890000,46.300000,19.200000,44.626667,19.790000,44.933333,18.926667,45.890000,17.166667,...,48.730000,17.000000,45.5000,6.366667,733.7,92.000000,6.333333,55.333333,5.100000,17
3,40,19.890000,46.066667,19.200000,44.590000,19.790000,45.000000,18.890000,45.723333,17.166667,...,48.590000,17.000000,45.4000,6.250000,733.8,92.000000,6.000000,51.500000,5.000000,17
4,40,19.890000,46.333333,19.200000,44.530000,19.790000,45.000000,18.890000,45.530000,17.200000,...,48.590000,17.000000,45.4000,6.133333,733.9,92.000000,5.666667,47.666667,4.900000,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19730,0,25.566667,46.560000,25.890000,42.025714,27.200000,41.163333,24.700000,45.590000,23.200000,...,50.074000,23.200000,46.7900,22.733333,755.2,55.666667,3.333333,23.666667,13.333333,17
19731,0,25.500000,46.500000,25.754000,42.080000,27.133333,41.223333,24.700000,45.590000,23.230000,...,49.790000,23.200000,46.7900,22.600000,755.2,56.000000,3.500000,24.500000,13.300000,17
19732,10,25.500000,46.596667,25.628571,42.768571,27.050000,41.690000,24.700000,45.730000,23.230000,...,49.660000,23.200000,46.7900,22.466667,755.2,56.333333,3.666667,25.333333,13.266667,17
19733,10,25.500000,46.990000,25.414000,43.036000,26.890000,41.290000,24.700000,45.790000,23.200000,...,49.518750,23.200000,46.8175,22.333333,755.2,56.666667,3.833333,26.166667,13.233333,17


In [5]:
# Train_Test_Split

X_train , X_test , y_train , y_test = train_test_split(
    X ,
    y ,
    test_size = 0.2 ,
    random_state = 42
)

In [36]:
## 1st Try

model = XGBRegressor(
    subsample = 0.8 ,
    colsample_bytree = 0.8 ,
    random_state = 42
)

paramm_grid = {
    "n_estimators" : randint(200 , 401),
    "max_depth" : randint(3 , 7) ,
    "learning_rate" : uniform(0.01 , 0.29),
    "gamma" : uniform(0,5), 
    "reg_lambda" : uniform(0,30),
    "reg_alpha" : uniform(0,10),
    "min_child_weight" : [1,2,3]
}

srch = RandomizedSearchCV(
    estimator = model ,
    param_distributions = paramm_grid ,
    cv = 5 ,
    n_iter = 15 ,
    n_jobs = -1
)

srch.fit(
    X_train ,
    y_train
)

model = srch.best_estimator_

y_pred = model.predict(X_test)

print(srch.best_params_)
print()
print(f"R2 Score : {r2_score(y_test , y_pred)}")

{'gamma': np.float64(0.020307154516508996), 'learning_rate': np.float64(0.2766887513553133), 'max_depth': 6, 'min_child_weight': 3, 'n_estimators': 321, 'reg_alpha': np.float64(9.75600594452365), 'reg_lambda': np.float64(17.086270906170295)}

R2 Score : 0.6572234153159004


# Runned this cell multiple times until I get best R2 Score 

# Considering max_depth = 6 & n_estimators = 321  & rearranging the other parametrs accordingly

In [9]:
# 2nd Try

model = XGBRegressor(
    subsample = 0.8 ,
    colsample_bytree = 0.8 ,
    random_state = 42 ,
    n_estimators = 321 ,
    max_depth = 6 ,
)

paramm_grid = {
    "learning_rate" : uniform(0.25 , 0.15),
    "gamma" : uniform(0 , 1), 
    "reg_lambda" : uniform(15 ,15),
    "reg_alpha" : uniform(7 , 15),
    "min_child_weight" : [3,4,5]
}

srch = RandomizedSearchCV(
    estimator = model ,
    param_distributions = paramm_grid ,
    cv = 5 ,
    n_iter = 15 ,
    n_jobs = -1
)

srch.fit(
    X_train ,
    y_train
)

model = srch.best_estimator_

y_pred = model.predict(X_test)

print(srch.best_params_)
print()
print(f"R2 Score : {r2_score(y_test , y_pred)}")

{'gamma': np.float64(0.04201548534960653), 'learning_rate': np.float64(0.37770545125717053), 'min_child_weight': 4, 'reg_alpha': np.float64(7.367859329843823), 'reg_lambda': np.float64(23.136594345558095)}

R2 Score : 0.6704924536587717


# Runned this cell multiple times until I get best R2 Score 

# Considering min_child_weight = 4 & rearranging the other parametrs accordingly

In [25]:
# 3rd Try

model = XGBRegressor(
    subsample = 0.8 ,
    colsample_bytree = 0.8 ,
    random_state = 42 ,
    n_estimators = 321 ,
    max_depth = 6 ,
    min_child_weight = 4
)

paramm_grid = {
    "learning_rate" : uniform(0.35 , 0.10),
    "gamma" : uniform(0.02 , 0.98), 
    "reg_lambda" : uniform(20 , 20),
    "reg_alpha" : uniform(5 , 10),
}

srch = RandomizedSearchCV(
    estimator = model ,
    param_distributions = paramm_grid ,
    cv = 5 ,
    n_iter = 15 ,
    n_jobs = -1
)

srch.fit(
    X_train ,
    y_train
)

model = srch.best_estimator_

y_pred = model.predict(X_test)

print(srch.best_params_)
print()
print(f"R2 Score : {r2_score(y_test , y_pred)}")

{'gamma': np.float64(0.03723250405834861), 'learning_rate': np.float64(0.4228618788460923), 'reg_alpha': np.float64(5.099320798875894), 'reg_lambda': np.float64(39.44256296205449)}

R2 Score : 0.6816550374579862


# Final Verdict
# ----------------------------------------------------------------------

# max_depth = 6
# n_estimators = 321
# min_child_weight = 4
# gamma = 0.0372
# learning_rate = 0.4228
# reg_alpha = 5.0993
# reg_lambda = 39.4425